In [3]:
import pandas as pd
import numpy as np

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

import chromadb

In [5]:
df = pd.read_csv("../data/raw/filtered_complaints.csv")

print("Shape:", df.shape)
df.head()

Shape: (2100, 13)


,complaint_id,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company,state,product_category,narrative,clean_narrative,word_count
0,10001645,2023-06-12,Ally Savings,NaN,Unauthorized access,Security breach,Synchrony charged me a $50 monthly maintenance...,Synchrony,CT,Savings Account,Synchrony charged me a $50 monthly maintenance...,synchrony charged me a 50 monthly maintenance ...,58
1,10000510,2024-01-10,Chase Sapphire,NaN,Rewards not credited,Sign-up bonus not received,Someone made 6 fraudulent purchases on my Chas...,American Express,UT,Credit Card,Someone made 6 fraudulent purchases on my Chas...,someone made 6 fraudulent purchases on my chas...,57
2,10001411,2024-12-07,Chase Savings,NaN,Unauthorized access,Security breach,Ally Bank charged me a $100 monthly maintenanc...,Ally Bank,MD,Savings Account,Ally Bank charged me a $100 monthly maintenanc...,ally bank charged me a 100 monthly maintenance...,59
3,10000045,2025-09-13,Capital One Venture,NaN,Credit limit reduction,Reduced without notice,The rewards program on my Capital One Venture ...,Capital One,MA,Credit Card,The rewards program on my Capital One Venture ...,the rewards program on my capital one venture ...,58
4,10001585,2023-09-26,Ally Savings,NaN,Account frozen,Frozen without warning,Synchrony closed my Ally Savings account witho...,Synchrony,CO,Savings Account,Synchrony closed my Ally Savings account witho...,synchrony closed my ally savings account witho...,50


In [6]:
df.columns.tolist()

['complaint_id',
 'date_received',
 'product',
 'sub_product',
 'issue',
 'sub_issue',
 'consumer_complaint_narrative',
 'company',
 'state',
 'product_category',
 'narrative',
 'clean_narrative',
 'word_count']

In [7]:
df["product_category"].value_counts()

product_category
Credit Card        800
Savings Account    600
Money Transfer     400
Personal Loan      300
Name: count, dtype: int64

Create a Stratified Sample

In [9]:
sampled_df = df.copy()

print("Dataset Shape:", sampled_df.shape)

sampled_df["product_category"].value_counts()

Dataset Shape: (2100, 13)


product_category
Credit Card        800
Savings Account    600
Money Transfer     400
Personal Loan      300
Name: count, dtype: int64

In [10]:
sampled_df.to_csv(
    "../data/raw/sample_complaints.csv",
    index=False
)

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = []

for _, row in sampled_df.iterrows():

    text = str(row["clean_narrative"])

    split_texts = splitter.split_text(text)

    for idx, chunk in enumerate(split_texts):

        chunks.append({
            "complaint_id": row["complaint_id"],
            "product_category": row["product_category"],
            "issue": row["issue"],
            "chunk": chunk,
            "chunk_index": idx,
            "total_chunks": len(split_texts)
        })

chunks_df = pd.DataFrame(chunks)

print("Total Chunks:", len(chunks_df))
chunks_df.head()

Total Chunks: 2100


,complaint_id,product_category,issue,chunk,chunk_index,total_chunks
0,10001645,Savings Account,Unauthorized access,synchrony charged me a 50 monthly maintenance ...,0,1
1,10000510,Credit Card,Rewards not credited,someone made 6 fraudulent purchases on my chas...,0,1
2,10001411,Savings Account,Unauthorized access,ally bank charged me a 100 monthly maintenance...,0,1
3,10000045,Credit Card,Credit limit reduction,the rewards program on my capital one venture ...,0,1
4,10001585,Savings Account,Account frozen,synchrony closed my ally savings account witho...,0,1


In [12]:
chunks_df.shape

(2100, 6)

Load the Embedding Model

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Acer\Documents\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


Test Embedding Generation

In [14]:
sample_embedding = model.encode(
    chunks_df["chunk"].iloc[0]
)

print(type(sample_embedding))
print(sample_embedding.shape)

<class 'numpy.ndarray'>
(384,)


Generate Embeddings for All Chunks

In [15]:
chunks_df["embedding"] = chunks_df["chunk"].apply(
    lambda text: model.encode(text).tolist()
)

print("Embeddings generated successfully!")

Embeddings generated successfully!


Verify Embeddings

In [16]:
chunks_df.head()

,complaint_id,product_category,issue,chunk,chunk_index,total_chunks,embedding
0,10001645,Savings Account,Unauthorized access,synchrony charged me a 50 monthly maintenance ...,0,1,"[-0.05444719269871712, 0.022850757464766502, 0..."
1,10000510,Credit Card,Rewards not credited,someone made 6 fraudulent purchases on my chas...,0,1,"[-0.04195297509431839, 0.044696081429719925, 0..."
2,10001411,Savings Account,Unauthorized access,ally bank charged me a 100 monthly maintenance...,0,1,"[-0.0009632958681322634, 0.0072071789763867855..."
3,10000045,Credit Card,Credit limit reduction,the rewards program on my capital one venture ...,0,1,"[-0.017349522560834885, -0.01250169612467289, ..."
4,10001585,Savings Account,Account frozen,synchrony closed my ally savings account witho...,0,1,"[-0.008269473910331726, 0.023799002170562744, ..."


In [17]:
len(chunks_df["embedding"].iloc[0])

384

Save Chunked Data with Embeddings

In [18]:
chunks_df.to_parquet(
    "../data/processed/chunks_with_embeddings.parquet",
    index=False
)

print("Saved successfully!")

Saved successfully!


Create ChromaDB Vector Store

In [19]:
import chromadb

client = chromadb.PersistentClient(
    path="../vector_store"
)

collection = client.get_or_create_collection(
    name="complaints"
)

Store Chunks in ChromaDB

In [20]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks_df))],
    documents=chunks_df["chunk"].tolist(),
    embeddings=chunks_df["embedding"].tolist(),
    metadatas=[
        {
            "complaint_id": str(row["complaint_id"]),
            "product_category": str(row["product_category"]),
            "issue": str(row["issue"]),
            "chunk_index": int(row["chunk_index"])
        }
        for _, row in chunks_df.iterrows()
    ]
)

print("Vector store created successfully!")

Vector store created successfully!


Test Semantic Search

In [21]:
query = "Customers are unhappy with credit card billing"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

results["documents"][0]

['i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.',
 'i noticed a charge of 750 on my chase sapphire statement from 2024-02-26. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they

Display Retrieved Results

In [22]:
for i, doc in enumerate(results["documents"][0], start=1):
    print(f"\n--- Result {i} ---")
    print(doc[:500])


--- Result 1 ---
i noticed a charge of 250 on my chase sapphire statement from 2025-05-07. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.

--- Result 2 ---
i noticed a charge of 2500 on my chase sapphire statement from 2025-10-12. i did not authorize this transaction and have been trying to resolve it with american express for 3 weeks. they keep transferring me between departments and no one takes responsibility. i have provided documentation proving i was not at the merchant location. this is extremely frustrating and i need this resolved immediately.

--- Result 3 ---
i noticed a charge of 750 on my chase sapphire statement from 2024-02-26. i did not authorize this transaction and have been trying to resolv

Display Metadata

In [23]:
results["metadatas"][0]

[{'chunk_index': 0,
  'issue': 'Account closure',
  'product_category': 'Credit Card',
  'complaint_id': '10000708'},
 {'issue': 'Account closure',
  'product_category': 'Credit Card',
  'chunk_index': 0,
  'complaint_id': '10000567'},
 {'chunk_index': 0,
  'product_category': 'Credit Card',
  'complaint_id': '10000499',
  'issue': 'Rewards not credited'},
 {'product_category': 'Credit Card',
  'complaint_id': '10000038',
  'issue': 'Billing dispute',
  'chunk_index': 0},
 {'chunk_index': 0,
  'product_category': 'Credit Card',
  'issue': 'Billing dispute',
  'complaint_id': '10000340'}]

## Task 2 Summary

- Loaded the cleaned CFPB complaint dataset.
- Used the filtered complaint records for the four target product categories.
- Applied RecursiveCharacterTextSplitter with:
  - Chunk Size: 500
  - Chunk Overlap: 50
- Generated semantic embeddings using all-MiniLM-L6-v2.
- Stored embeddings and metadata in a ChromaDB vector store.
- Successfully tested semantic retrieval using similarity search queries.